# GPU Profiling

1. Initialize CUDA, select the GPU, and configure NVML for hardware monitoring.
2. Create three `threading.Event` flags to track matrix multiplication, checkpointing, and job completion.
3. Start a separate monitoring thread that samples GPU utilization, memory, and power using `nvidia-smi`, PyTorch, and NVML.
4. Sample every 10 ms (target interval) and write measurements and activity flags to `gpu_metrics.csv`.
5. Run five iterations, each performing 1,000 GPU matrix multiplications, followed by CUDA synchronization.
6. Save GPU tensors to disk 100 times per iteration to simulate checkpointing overhead.
7. Measure execution time, exclude the first iteration from steady-state timing, and pause 1.1 seconds between iterations.
8. Stop the monitoring thread and visualize GPU metrics alongside computation and checkpoint activity using Plotly.

- https://developer.nvidia.com/management-library-nvml
- https://developer.nvidia.com/cuda/toolkit
- https://docs.pytorch.org/docs/2.13/cuda.html
- https://pypi.org/project/nvidia-ml-py/

### Imports and Colab dependency
Installs `nvidia-ml-py` for Colab, then keeps the original imports unchanged.


In [1]:
%pip install -q nvidia-ml-py

In [2]:
import csv, time, os, subprocess, threading
import torch, pynvml # we will use pynvml to query the GPU metrics
import os, pandas as pd, plotly.graph_objects as go
import shutil

### `nvsmi_query`
Queries GPU utilization, memory usage, total memory, and power draw through `nvidia-smi`.


In [3]:
def nvsmi_query(phys_index: int):
    out = subprocess.check_output( # we use subprocess to call nvidia-smi
        ["nvidia-smi",
         "-i", str(phys_index), # we specify the physical index of the GPU
         "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw", # here we query the GPU utilization, memory used, memory total, and power draw
         "--format=csv,noheader,nounits"], # we format the output as a CSV file
        encoding="utf-8")
    line = out.strip().splitlines()[0] # we passed -i, so we will get only one line of output
    util, mem_used, mem_total, power = [x.strip() for x in line.split(",")] # we split the line into the utilization, memory used, memory total, and power draw
    return int(util), float(mem_used), float(mem_total), float(power) # we return them accordingly


### `task_matmul`
Runs repeated CUDA matrix multiplication work and marks the active period.


In [4]:
def task_matmul(device, n, is_matmul_working):
    a = torch.randn((n, n), device=device)
    b = torch.randn((n, n), device=device)
    is_matmul_working.set()  # mark as "busy"
    for j in range(1000):
        c = a @ b # we perform the matrix multiplication
    # Without a sync, t1 - t0 measures “time to enqueue the kernel”, not “time for the matmul to actually finish”. torch.cuda.synchronize() blocks the CPU until all queued GPU work on that device is finished. So for timing / profiling:
    torch.cuda.synchronize()  # wait until all queued CUDA kernels on this device have finished
    is_matmul_working.clear()  # mark as "idle"
    del a, b, c # we delete the matrices to free up memory
    torch.cuda.empty_cache() # we empty the cache to free up memory


### `task_checkpoints`
Writes GPU tensor checkpoints. The only code change here adds `/content/checkpoints` as the Colab-safe default directory.


In [5]:
def task_checkpoints(device, n, i, is_checkpoints_working):
    ckpt_dir = os.environ.get("CHECKPOINT_DIR", "/content/checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    a = torch.randn((n, n), device=device)
    is_checkpoints_working.set()  # mark as "busy"
    for j in range(100):
        torch.save(a, os.path.join(ckpt_dir, "checkpoint.pt"))
    is_checkpoints_working.clear()  # mark as "idle"
    del a # In Python, del just removes a reference from the current scope. PyTorch tensors hold CUDA memory through their underlying storage.When the reference count drops to zero and the object is destroyed, PyTorch's CUDA allocator marks that chunk as free (but see caching below). When PyTorch “deletes” a CUDA tensor (refcount hits 0), it frees it only inside PyTorch’s own allocator. But PyTorch's allocator is caching: it keeps that GPU memory for itself so it can reuse it quickly later, instead of giving it back to the driver every time.
    torch.cuda.empty_cache() # we empty the cache to free up memory


### `worker_job`
Runs the matrix multiplication and checkpoint tasks for each iteration and records steady-state execution time.


In [6]:
def worker_job(device, n, iters, idle_gap, is_matmul_working, is_checkpoints_working):
    total_clean_time = 0.0
    for i in range(iters):
        t0worker_job = time.perf_counter() # we start the timer
        task_matmul(device, n, is_matmul_working)
        task_checkpoints(device, n, i, is_checkpoints_working)
        t1worker_job = time.perf_counter() # we stop the timer
        print(f"worker_job {i} took {t1worker_job - t0worker_job:.10f}s", flush=True) # we print the time taken for the matrix multiplication
        if i > 0: # skip the first iteration: it often includes one-time CUDA/cuBLAS initialization, so we only accumulate steady-state worker_job time.
            total_clean_time += t1worker_job - t0worker_job
        time.sleep(idle_gap)      # real idle time so nvidia-smi/NVML can see the drop
    return total_clean_time # we return the total time taken for the matrix multiplications


### `parallel_sampler_thread`
Samples GPU metrics in parallel and writes the measurements to CSV.


In [7]:
def parallel_sampler_thread(device, handle, csv_file, interval_target_dt, is_matmul_working, is_checkpoints_working, has_job_finished, phys_index):
    with open(csv_file, "w", newline="") as f: # Initialize the CSV file
        w = csv.writer(f)
        w.writerow(["timestamp","dt","is_matmul_working","is_checkpoints_working",
                    "GPU-Util-SMI","MEM-Used-SMI","Power-SMI",
                    "GPU-Util-Torch","MEM-Used-Torch","Power-Torch",
                    "GPU-Util-pyNVML","MEM-Used-pyNVML","Power-pyNVML"])
        # NOTE: GPU utilization (from NVML / torch.cuda.utilization) is a trailing
        # average over roughly the last 1 second of activity, not an instantaneous
        # value. That means these samples will lag behind the is_matmul_working flag
        # and stay high for a short time after the worker goes idle. This is expected.
        t_initial = time.perf_counter()
        t_end = time.perf_counter()
        while not has_job_finished.is_set():
            t_start = time.perf_counter()
            delta_t = t_start - t_end
            t_end = time.perf_counter()
            n_util, n_mem, n_total_mem, n_power = nvsmi_query(phys_index) # we query the GPU metrics using nvidia-smi
            power_limit_mw = pynvml.nvmlDeviceGetPowerManagementLimit(handle)
            # torch_mem = torch.cuda.memory_allocated(device) / (1024**2)  # NOT used: only counts live PyTorch tensors, ignores cached blocks, CUDA context/driver overhead, and other processes.
            free_b, total_b = torch.cuda.memory.mem_get_info(device)
            torch_mem = (total_b - free_b) / total_b  # use global device usage, comparable to NVML / nvidia-smi (includes allocator cache, context, and other processes).
            torch_util = torch.cuda.utilization(device) # we get the gpu utilization by PyTorch
            torch_power = torch.cuda.power_draw(device) / power_limit_mw # we get the power draw by PyTorch
            util = pynvml.nvmlDeviceGetUtilizationRates(handle) # we get the gpu utilization by NVML
            mem = pynvml.nvmlDeviceGetMemoryInfo(handle) # we get the memory usage by NVML. CUDA (mem_get_info) and nvidia-smi report ~same MiBs, while NVML’s mem.used can report higher MiBs because it includes additional driver-reserved memory and possibly other allocations not counted by mem_get_info
            p_power = pynvml.nvmlDeviceGetPowerUsage(handle) / power_limit_mw # we get the power draw by NVML
            p_mem = mem.used / mem.total # we convert the memory usage to 0-1 ratio
            w.writerow([t_start-t_initial,delta_t,int(is_matmul_working.is_set()),int(is_checkpoints_working.is_set()),
                        n_util/100.0,n_mem/n_total_mem,1000*n_power/power_limit_mw,
                        torch_util/100.0,torch_mem,torch_power,
                        util.gpu/100.0,p_mem,p_power])
            f.flush()
            elapsed_dt = time.perf_counter() - t_start
            sleep_time = interval_target_dt - elapsed_dt # we sleep for the remaining time to reach the next sampling interval_target_dt.
            if sleep_time > 0: time.sleep(sleep_time)


### `main`
Selects the CUDA device, initializes NVML, starts sampling, runs the workload, and shuts down cleanly.


In [8]:
def main():
    if not torch.cuda.is_available(): raise SystemExit("CUDA is not available")
    torch_index = torch.cuda.current_device()  # logical device index (after CUDA_VISIBLE_DEVICES)
    device = torch.device(f"cuda:{torch_index}")  # use the current GPU
    n = 2_000 # the size of the matrices for the matrix multiplication.
    worker_job_iters = 5 # the number of matrix multiplications to perform.
    interval_target_dt = 0.01    # parallel_sampler_thread interval_target_dt: how often to sample the GPU metrics. Note: interval_target_dt=0.01s + nvidia-smi/NVML is heavy; this is for small demos and can perturb real training timing.
    idle_gap = 1.1   # idle time between worker_jobs: Make the idle gap longer than the NVML / nvidia-smi sampling window, which is on the order of 1 second, so at least one whole sample sees no kernels running and reports low utilization. Change the value back to something > 1.0 if you really want to see idle periods in NVML/SMI. https://docs.nvidia.com/deploy/nvidia-smi/index.html
    print(f"Parameters: n={n}, worker_job_iters={worker_job_iters}, interval_target_dt={interval_target_dt}s, idle_gap={idle_gap}s")
    csv_file = os.environ.get("GPU_METRICS_CSV", "gpu_metrics.csv")
    print("Using device:", device)
    # Map logical CUDA index -> physical NVML index using CUDA_VISIBLE_DEVICES (if set)
    visible = os.environ.get("CUDA_VISIBLE_DEVICES")  # read CUDA-visible GPU list, e.g. "2,3" or "0"
    if visible:  # if CUDA_VISIBLE_DEVICES is set, logical indices are remapped
        phys_list = [int(x) for x in visible.split(",")]  # convert "2,3" -> [2, 3]
        phys_index = phys_list[torch_index]  # map logical torch_index (0,1,...) to physical GPU ID
    else:
        phys_index = torch_index  # no remap: logical index == physical index
    pynvml.nvmlInit() # initialize the NVML library.
    handle = pynvml.nvmlDeviceGetHandleByIndex(phys_index) # get the handle for the current GPU.
    # Use threading.Event instead of plain variables so the flags actually communicate between threads.
    # Both parallel_sampler_thread and worker_job share the same Event objects (passed in as arguments here because they are created in main(), but they could also be globals).
    # When one thread sets or clears an Event, the other can observe the change via is_set().
    is_matmul_working = threading.Event()
    is_checkpoints_working = threading.Event()
    has_job_finished = threading.Event()
    t_parallel_sampler_thread = threading.Thread(target=parallel_sampler_thread,
                                 args=(device, handle, csv_file, interval_target_dt, is_matmul_working, is_checkpoints_working, has_job_finished, phys_index),
                                 daemon=True)
    t_parallel_sampler_thread.start() # start the parallel_sampler_thread thread.
    time.sleep(idle_gap) # we sleep for the extra idle logging duration.
    t_start = time.perf_counter()
    total_clean_time = worker_job(device, n, worker_job_iters, idle_gap, is_matmul_working, is_checkpoints_working) # actually perform the worker_job.
    t_end = time.perf_counter()
    print(f"worker_job worker finished in {t_end - t_start:.3f}s, total clean time: {total_clean_time:.10f}s")
    time.sleep(idle_gap) # we sleep for the extra idle logging duration.
    has_job_finished.set()  # signal parallel_sampler_thread to stop
    t_parallel_sampler_thread.join() # We wait for the parallel_sampler_thread thread to finish.
    pynvml.nvmlShutdown() # we shutdown the NVML library.
    print("All Done.", flush=True)

### Run the program
Calls `main()` exactly as in the original script.


In [9]:
if __name__ == "__main__": main()

Parameters: n=2000, worker_job_iters=5, interval_target_dt=0.01s, idle_gap=1.1s
Using device: cuda:0
worker_job 0 took 12.1661395620s
worker_job 1 took 11.6895016760s
worker_job 2 took 11.7044479250s
worker_job 3 took 11.8770780190s
worker_job 4 took 11.9982165310s
worker_job worker finished in 64.945s, total clean time: 47.2692441510s
All Done.


### Plot the recorded GPU metrics

In [10]:
csv_file = os.environ.get("GPU_METRICS_CSV", "gpu_metrics.csv") # same default as main
frame = pd.read_csv(csv_file) # read the sampler output
metric_columns = ["GPU-Util-SMI","GPU-Util-Torch","GPU-Util-pyNVML",
                  "MEM-Used-SMI","MEM-Used-Torch","MEM-Used-pyNVML",
                  "Power-SMI","Power-Torch","Power-pyNVML"] # all columns are 0-1 ratios
figure = go.Figure()
for column_name in metric_columns:
    figure.add_trace(go.Scatter(x=frame["timestamp"], y=frame[column_name], name=column_name, mode="lines"))
figure.add_trace(go.Scatter(x=frame["timestamp"], y=frame["is_matmul_working"], name="matmul busy", mode="lines", line=dict(dash="dot")))
figure.add_trace(go.Scatter(x=frame["timestamp"], y=frame["is_checkpoints_working"], name="ckpt busy", mode="lines", line=dict(dash="dot")))
figure.update_layout(xaxis_title="seconds", yaxis_title="ratio 0-1", height=650, hovermode="x unified")
figure.show()